## About
- This notebook includes evaluation of prioritized models at predicting steatohepatitis at risk of progression

### Input
> Previously prioritized 2-feature prediction models for detecting steatohepatitis

### Output
> Top 10 models based on AUROC in the test set

### Steatohepatitis at risk of progression 

- "steatohepatitis at risk of progression" (≥NAS4 & ≥F2) with FAST score as comparator: 
- https://doi.org/10.1016%2FS2468-1253(19)30383-8 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import pickle
from sklearn.metrics import roc_auc_score, roc_curve
from src.plots import plot_roc_curve
from sklearn.model_selection import train_test_split
from datetime import datetime
from tqdm import tqdm 
import feyn
import ast
import pickle

In [ ]:
# Define project directory
Base = 'data_directory'

In [ ]:
# Define project directory
df_olink = pd.read_excel(os.path.join(Base, 'data_curated/cureated_proteomics_clinical_data.xlsx'), sheet_name='proteomics_curated').set_index('CBMRID')
df_cli = pd.read_excel(os.path.join(Base, 'data_curated/cureated_proteomics_clinical_data.xlsx'), sheet_name='clinical_curated').set_index('CBMRID')

# Join curated proteomics and clinical data
data = df_olink.join(df_cli)
# Drop samples without outcome target measure
data = data[data['cohort']!='GALA_LiHep'].dropna(subset=['inflam_binary'])

# import MS data 
data_ms = pd.read_csv(os.path.join(Base, 'data_curated/olink_ms_combined.csv')).set_index('CBMRID')
data=data.join(data_ms[['Q08380']])

In [ ]:
data_atrisk = data.dropna(subset=['at_risk_progression', 'ast', 'alt', 'fast'])

In [ ]:
data['at_risk_progression'].count()

In [ ]:
data_atrisk.groupby('cohort')['at_risk_progression'].value_counts(1)

#### Cohorts

In [ ]:
rdc_ids=data[data['cohort']=='RDC'].index
ald_ids=data[data['cohort']=='ALD'].index
hp_ids=data[data['cohort']=='HP'].index
sip_ids=data[data['cohort']=='SIP'].index
lihep_ids = data[data['cohort']=='GALA_LiHep'].index
sip_biop_ids = data.iloc[data.index.isin(sip_ids)][['lobinflammation', 'ballooning']].dropna().index
ikkesupernomral_ids = data[data['supernormal_filter']].index

In [ ]:
_train_ar = data_atrisk[data_atrisk.index.isin(ald_ids.tolist() + hp_ids.tolist() + rdc_ids.tolist())]
train_ar, test_ar = train_test_split(_train_ar, random_state=42, test_size=.2, stratify=_train_ar['at_risk_progression'])
validation1_ar = data_atrisk[data_atrisk.index.isin(sip_ids)]

In [ ]:
best5 = ['fast',
         ('HGF', 'ast_log2'), 
         ('HGF', 'SCF'), 
         ('HGF', 'TWEAK'), 
         ('SCF', 'SLAMF1'),
         ('HGF', 'TRAIL'),
         ]

In [ ]:
from sklearn.metrics import confusion_matrix, roc_auc_score, f1_score, precision_score, recall_score, accuracy_score, balanced_accuracy_score, matthews_corrcoef

def get_scoring(y_true, y_score, y_pred):
    scores = []
    cm = confusion_matrix(y_true, y_pred)
    tn = cm[0][0]
    fn = cm[1][0]
    npv = tn/(tn+fn)
    roc_auc = roc_auc_score(y_true, y_score)
    f1 = f1_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    accuracy = accuracy_score(y_true, y_pred)
    balanced_accuracy = balanced_accuracy_score(y_true, y_pred)
    specificity = 2 * balanced_accuracy - recall
    mcc = matthews_corrcoef(y_true, y_pred) 
    scores = [roc_auc, f1, precision, npv, recall, accuracy, balanced_accuracy, specificity, mcc]
    names = ['roc_auc', 'f1', 'precision','NPV', 'recall', 'accuracy', 'balanced accuracy', 'specificity', 'mcc']
    scores = pd.DataFrame(dict(zip(names, scores)), index=[0])
    return scores.round(3)

In [ ]:
file_atrisk_models = os.path.join(Base, 'symbolic_regression/models/atrisk_models.pkl')

RE_COMPUTE = False
if not RE_COMPUTE:
    with open(file_atrisk_models, 'rb') as f:
        combo_models_atrisk = pickle.load(f)
else:        
    combo_models_atrisk = {}
    n=1
    with open('at_risk/log/run_combo_at_risk.log', 'w') as file:
        now=datetime.now()
        current_time=now.strftime("%D %H:%M:%S")
        file.write("Analysis started at {}\n".format(current_time))
        file.flush()
        for combo in tqdm(best5):
            if combo == 'fast':
                selected_features = [combo]
            else:
                selected_features = list(combo)
            file.write("combo{}: {}\n".format(n, selected_features))
            file.flush()
            print("combo{}: {}".format(n, selected_features))
            n+=1
            train_combo_atrisk = train_ar[selected_features + ['at_risk_progression']].dropna()
            ql = feyn.QLattice(random_seed=42)
            models = ql.auto_run(
                data=train_combo_atrisk,
                output_name='at_risk_progression',
                query_string = "_{}".format(selected_features),
                kind='classification'
            )
            combo_models_atrisk[combo]=models[0]
        now=datetime.now()
        current_time=now.strftime("%D %H:%M:%S")
        file.write('Analysis done at {}'.format(current_time))
        file.flush()
    
    with open(file_atrisk_models, 'wb') as f:
        pickle.dump(combo_models_atrisk, f)

In [ ]:
model_performance_atrisk= []
roc_curves_combo = {}
for feature in combo_models_atrisk.keys():
    model = combo_models_atrisk[feature]
    formula = str(model.sympify(symbolic_lr=True, include_weights=True))
    features = model.features
    datasets = {"train":train_ar, "test": test_ar, 
                'validation1':validation1_ar}
    roc_curves = {}
    for split in ["train", "test", "validation1"]:
        dataset_split=datasets[split].dropna(subset=['at_risk_progression'])
        y_true = dataset_split['at_risk_progression']
        y_score = model.predict(dataset_split)
        y_pred = y_score.round()
        roc_curves[split]=(y_true, y_score, y_pred)
        nr_obs=len(y_pred)
        class1_pert = y_true.sum()/nr_obs*100
        scoring = get_scoring(y_true, y_score, y_pred)
        scoring['nr_obs']=nr_obs
        scoring['class1_pert']=class1_pert.astype(int)
        scoring['set']=split
        scoring['features']=str(features)
        scoring['formula']=formula
        scoring['feature1']=ast.literal_eval(str(features))[0]
        if len(features)==1:
            scoring['feature2']=ast.literal_eval(str(features))[0]
        else:
            scoring['feature2']=ast.literal_eval(str(features))[1]
        scoring['nr_features']=len(features)
        scoring['model']=str(model)
        model_performance_atrisk.append(scoring)
    roc_curves_combo[feature]=roc_curves
model_performance_atrisk = pd.concat(model_performance_atrisk)

In [ ]:
df_final = model_performance_atrisk.copy()

In [ ]:
df_final[df_final['set']=='validation1'].sort_values(by= 'roc_auc', ascending=False)

#### Bootstrap to calculate confidence intervals at 2.5th and 97.5th

In [ ]:
top5 = [('HGF', 'SCF'), 
       ('HGF', 'TWEAK'),
       ('HGF', 'TRAIL'),
       ('SCF', 'SLAMF1'),
       ('HGF', 'ast_log2'),
       'fast',]

from sklearn.utils import resample
n_boot = 1000
boot_all = []
target = 'at_risk_progression'

for combo in tqdm(top5):
    boot_res = []
    model = combo_models_atrisk[combo]
       
    for i in range(n_boot):
    
        # Bootstrap validation cohort
        boot_df = resample(
            validation1_ar,
            replace=True,
            n_samples=len(validation1_ar),
            random_state=i,
            stratify=validation1_ar[target]
        )
    
        # Predictions
        y_true = boot_df[target]
        y_score = model.predict(boot_df)
        y_pred = y_score.round()
    
        # Compute all metrics
        scoring = get_scoring(y_true, y_score, y_pred)
        boot_res.append(scoring)
    
    df_res = pd.concat(boot_res)
    
    ci = pd.DataFrame({
        "estimate": df_res.mean(),
        "lower95": df_res.quantile(0.025),
        "upper95": df_res.quantile(0.975)
    })
    ci_model = ci.T.copy()
    ci_model['features']=str(combo)
    boot_all.append(ci_model)

In [ ]:
df_boot = pd.concat(boot_all)
df_boot.to_csv(os.path.join(Base, 'symbolic_regression/models/ci_estimate_atrisk.csv'))

### Write to results

In [ ]:
with pd.ExcelWriter('supp_tables/ST8.xlsx') as writer:
    df_final.to_excel(writer, sheet_name='ST8', index=False)

### Plot

In [ ]:
def plot_roc_curve(true_y, y_prob, color):
    """
    plots the roc curve based of the probabilities
    """

    fpr, tpr, thresholds = roc_curve(true_y, y_prob)
    plt.plot(fpr, tpr, color)
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')

In [ ]:
combo_labels = {
    'fast': 'FAST',
    ('HGF', 'ast_log2'): 'HGF + AST'
}

combos_toplot = ['fast', ('HGF', 'ast_log2')]
colors = ['gray', 'steelblue']
heights = [0.2, 0.1]

set_titles = {'train': 'Training set', 'test': 'Test set', 'validation1': 'Validation cohort'}

for set_toplot in ['train', 'test', 'validation1']:
    fig, ax = plt.subplots(figsize=(4, 4))
    for combo_toplot, color, height in zip(combos_toplot, colors, heights):
        y_true = roc_curves_combo[combo_toplot][set_toplot][0]
        y_prob = roc_curves_combo[combo_toplot][set_toplot][1]
        plot_roc_curve(y_true, y_prob, color=color)
        auroc = roc_auc_score(y_true, y_prob)
        label = combo_labels[combo_toplot]
        ax.text(0.45, height, '{} AUROC: {}'.format(label, round(auroc, 2)),
                color=color, fontsize=9, transform=ax.transAxes)

    ax.plot([0, 1], [0, 1], ls="--", c='lightgray')
    ax.set_xlabel('False Positive Rate', fontsize=10)
    ax.set_ylabel('True Positive Rate', fontsize=10)
    ax.set_title(set_titles[set_toplot], fontsize=11, fontweight='bold')
    ax.tick_params(labelsize=9)
    sns.despine(ax=ax)

    plt.rcParams['pdf.fonttype'] = 42
    plt.tight_layout()
    # plt.savefig('figures/roc_auc_atrisk_{}.pdf'.format(set_toplot), dpi=120, bbox_inches='tight')

In [ ]:
combo_labels = {
    'fast': 'FAST',
    ('HGF', 'ast_log2'): 'HGF + AST'
}

combos_toplot = ['fast', ('HGF', 'ast_log2')]
colors = ['gray', 'steelblue']
heights = [0.2, 0.1]

set_titles = {'train': 'Training set', 'test': 'Test set', 'validation1': 'Validation cohort'}

for set_toplot in ['train', 'test', 'validation1']:
    fig, ax = plt.subplots(figsize=(4, 4))
    for combo_toplot, color, height in zip(combos_toplot, colors, heights):
        y_true = roc_curves_combo[combo_toplot][set_toplot][0]
        y_prob = roc_curves_combo[combo_toplot][set_toplot][1]
        plot_roc_curve(y_true, y_prob, color=color)
        auroc = roc_auc_score(y_true, y_prob)
        label = combo_labels[combo_toplot]
        ax.text(0.45, height, '{} AUROC: {}'.format(label, round(auroc, 2)),
                color=color, fontsize=9, transform=ax.transAxes)

    ax.plot([0, 1], [0, 1], ls="--", c='lightgray')
    ax.set_xlabel('False Positive Rate', fontsize=10)
    ax.set_ylabel('True Positive Rate', fontsize=10)
    ax.set_title(set_titles[set_toplot], fontsize=11, fontweight='bold')
    ax.tick_params(labelsize=9)
    sns.despine(ax=ax)

    plt.rcParams['pdf.fonttype'] = 42
    plt.tight_layout()
    # plt.savefig('figures/roc_auc_atrisk_{}.pdf'.format(set_toplot), dpi=120, bbox_inches='tight')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

for ax, set_toplot, label in zip(axes, ['train', 'test', 'validation1'], 'abc'):
    plt.sca(ax)  # set current axis so plot_roc_curve draws on correct subplot
    for combo_toplot, color, height in zip(combos_toplot, colors, heights):
        y_true = roc_curves_combo[combo_toplot][set_toplot][0]
        y_prob = roc_curves_combo[combo_toplot][set_toplot][1]
        plot_roc_curve(y_true, y_prob, color=color)
        auroc = roc_auc_score(y_true, y_prob)
        ax.text(0.45, height, '{} AUROC: {}'.format(combo_labels[combo_toplot], round(auroc, 2)),
                color=color, fontsize=9, transform=ax.transAxes)

    ax.plot([0, 1], [0, 1], ls="--", c='lightgray')
    ax.set_xlabel('False Positive Rate', fontsize=10)
    ax.set_ylabel('True Positive Rate', fontsize=10)
    ax.set_title(set_titles[set_toplot], fontsize=11, fontweight='bold')
    ax.tick_params(labelsize=9)
    ax.text(-0.12, 1.07, label + '.', transform=ax.transAxes, fontsize=12, fontweight='bold', va='top')
    sns.despine(ax=ax)

plt.rcParams['pdf.fonttype'] = 42
plt.tight_layout()
plt.savefig('temp.pdf')

In [ ]:
combo_models_atrisk[('HGF', 'ast_log2')]

In [ ]:
combo_models_atrisk[('HGF', 'ast_log2')].sympify(symbolic_lr=True, include_weights=True)

In [ ]:
combo_models_atrisk['fast'].sympify(symbolic_lr=True, include_weights=True)

#### Agreement between FAST and INF-2

In [ ]:
pair = ('HGF', 'ast_log2')
set_toplot = 'validation1'
gt = pd.DataFrame(roc_curves_combo[pair][set_toplot][0])
for pair in combos_toplot:
    res = roc_curves_combo[pair][set_toplot][2]
    gt[pair]=res

In [ ]:
def categorize(row):
    c1_ok = row['fast'] == row['at_risk_progression']
    c2_ok = row[('HGF', 'ast_log2')] == row['at_risk_progression']
    if c1_ok and c2_ok:
        return 'Both correct'
    elif not c1_ok and not c2_ok:
        return 'Both wrong'
    elif c1_ok:
        return 'fast right only'
    else:
        return 'INF2 right only'

gt['concordance'] = gt.apply(categorize, axis=1)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, clf, title, ylabel in zip(
    axes[:2],
    ['fast', ('HGF', 'ast_log2')],
    ['FAST', 'INF2'],
    ['Classified by FAST', 'Classified by INF-2']
):
    cm = confusion_matrix(gt[clf], gt['at_risk_progression'])
    ConfusionMatrixDisplay(cm).plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_ylabel(ylabel)
    ax.set_xlabel('')

    # move x-axis to top
    ax.xaxis.set_label_position('top')
    ax.xaxis.tick_top()
    ax.set_xlabel('Ground Truth', labelpad=10)

gt['concordance'].value_counts().plot(kind='bar', ax=axes[2], color='steelblue', edgecolor='white')
axes[2].set_ylabel('Count')
axes[2].set_title('Concordance categories')
axes[2].tick_params(axis='x', rotation=30)

plt.rcParams['pdf.fonttype'] = 42
plt.tight_layout()
plt.savefig('figures/temp.pdf')

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, clf, title, ylabel in zip(
    axes[:2],
    ['fast', ('HGF', 'ast_log2')],
    ['FAST', 'INF2'],
    ['Classified by FAST', 'Classified by INF-2']
):
    cm = confusion_matrix(gt[clf], gt['at_risk_progression'])
    ConfusionMatrixDisplay(cm).plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_ylabel(ylabel)
    ax.set_xlabel('')

    # move x-axis to top
    ax.xaxis.set_label_position('top')
    ax.xaxis.tick_top()
    ax.set_xlabel('Ground Truth', labelpad=10)

gt['concordance'].value_counts().plot(kind='bar', ax=axes[2], color='steelblue', edgecolor='white')
axes[2].set_ylabel('Count')
axes[2].set_title('Concordance categories')
axes[2].tick_params(axis='x', rotation=30)

plt.rcParams['pdf.fonttype'] = 42
plt.tight_layout()
plt.savefig('temp.pdf')

In [ ]:
import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, clf, title, ylabel in zip(axes[:2], ['fast', ('HGF', 'ast_log2')], ['FAST', 'INF2'], ['FAST', 'INF-2']):
    ConfusionMatrixDisplay(confusion_matrix(gt[clf], gt['at_risk_progression'])).plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_ylabel(ylabel)
    ax.xaxis.set_label_position('top')
    ax.xaxis.tick_top()
    ax.set_xlabel('Ground Truth', labelpad=10)

order = ['Both correct', 'INF2 right only', 'Both wrong', 'fast right only']
neg = gt[gt['at_risk_progression'] == 0]['concordance'].value_counts().reindex(order, fill_value=0)
pos = gt[gt['at_risk_progression'] == 1]['concordance'].value_counts().reindex(order, fill_value=0)

axes[2].bar(range(4), neg, color='steelblue', label='GT=0')
axes[2].bar(range(4), pos, bottom=neg, color='lightsteelblue', label='GT=1')

for i, (n, p) in enumerate(zip(neg, pos)):
    if n > 0: axes[2].text(i, n/2,   str(n), ha='center', va='center', fontsize=9, color='white')
    if p > 0: axes[2].text(i, n+p/2, str(p), ha='center', va='center', fontsize=9, color='steelblue')

axes[2].set_xticks(range(4))
axes[2].set_xticklabels(order, rotation=30, ha='right')
axes[2].set_title('Concordance categories')
axes[2].legend(fontsize=9)
plt.tight_layout()
plt.savefig('temp.pdf', transparent=False, bbox_inches='tight')
plt.show()